# [UC1] Integración: Contexto JSON para Havi — Financial Copilot

**Owner:** Fernando Haro  
**Serie:** 30 (Integración)  
**Dependencia:** `outputs/uc1/uc1_trigger_metrics.json`, hey_transacciones.csv, hey_productos.csv, hey_clientes.csv

## Objetivo
Implementar `build_context_uc1(txn_id, user_id)` que genera el payload JSON de contexto para Havi,  
mockear `transferFunds()`, y demostrar 3 escenarios completos con datos reales del dataset.


In [1]:
import os
from pathlib import Path as _Path

# Navigate to repo root (works both in JupyterLab and nbconvert)
for _candidate in [_Path.cwd()] + list(_Path.cwd().parents):
    if (_candidate / "INTEGRATION.md").exists():
        os.chdir(_candidate)
        break
print("Working dir:", os.getcwd())


Working dir: C:\Users\Fernando\Documents\GitHub\Datathon-2026


In [2]:
import pandas as pd
import json
import re
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

BASE_TXN  = Path("Datathon_Hey_2026_dataset_transacciones 1/dataset_transacciones")
BASE_OUT  = Path("outputs/integration")
BASE_OUT.mkdir(parents=True, exist_ok=True)

df_tx   = pd.read_csv(BASE_TXN / "hey_transacciones.csv")
df_prod = pd.read_csv(BASE_TXN / "hey_productos.csv")
df_cli  = pd.read_csv(BASE_TXN / "hey_clientes.csv")

MOTIVOS_UC1 = ["saldo_insuficiente", "limite_excedido"]
df_rechazos = df_tx[
    (df_tx["estatus"] == "no_procesada") &
    (df_tx["motivo_no_procesada"].isin(MOTIVOS_UC1))
].copy()

print(f"Total rechazos UC1: {len(df_rechazos):,}")
print(f"Usuarios únicos:    {df_rechazos['user_id'].nunique():,}")
print(f"Motivo saldo:       {(df_rechazos['motivo_no_procesada']=='saldo_insuficiente').sum():,}")
print(f"Motivo límite:      {(df_rechazos['motivo_no_procesada']=='limite_excedido').sum():,}")


Total rechazos UC1: 6,638
Usuarios únicos:    5,079
Motivo saldo:       3,334
Motivo límite:      3,304


## Schema del Payload UC1

Estructura del JSON que se inyecta como contexto a Havi:

In [3]:
UC1_CONTEXT_SCHEMA = {
    "user_id":                      "str — identificador del usuario",
    "transaccion_id":               "str — ID de la txn rechazada",
    "situacion":                    "str — 'rechazo_por_saldo' | 'rechazo_por_limite'",
    "motivo":                       "str — motivo_no_procesada raw",
    "monto_rechazado":              "float — monto de la compra en MXN",
    "comercio":                     "str — nombre o categoría del comercio",
    "ciudad_transaccion":           "str — ciudad donde ocurrió el cargo",
    "fecha_hora":                   "str — timestamp de la transacción",
    "es_internacional":             "bool — si fue cargo en el extranjero",
    "nombre_usuario":               "str — nombre del cliente (para personalizar)",
    "saldo_actual_producto_origen": "float — saldo disponible en el producto rechazado",
    "monto_faltante":               "float — cuánto le faltó para completar la compra",
    "tiene_alternativo":            "bool — si existe producto alternativo con fondos",
    "producto_alternativo":         "str | None — tipo_producto del alternativo",
    "producto_alternativo_id":      "str | None — ID del producto alternativo",
    "monto_disponible_alternativo": "float | None — saldo en el producto alternativo",
    "es_cronico":                   "bool — si el usuario tiene ≥3 rechazos en 30 días",
    "canal_preferido":              "str — canal de contacto preferido",
}
print("Schema UC1 definido:", list(UC1_CONTEXT_SCHEMA.keys()))


Schema UC1 definido: ['user_id', 'transaccion_id', 'situacion', 'motivo', 'monto_rechazado', 'comercio', 'ciudad_transaccion', 'fecha_hora', 'es_internacional', 'nombre_usuario', 'saldo_actual_producto_origen', 'monto_faltante', 'tiene_alternativo', 'producto_alternativo', 'producto_alternativo_id', 'monto_disponible_alternativo', 'es_cronico', 'canal_preferido']


## `build_context_uc1(txn_id, user_id)`

In [4]:
PRIORIDAD_PRODUCTO = {
    "cuenta_debito":          1,
    "inversion_hey":          2,
    "tarjeta_credito_hey":    3,
    "cuenta_ahorro":          4,
    "credito_nomina":         5,
    "credito_personal":       6,
}
MONTO_MINIMO_MXN = 50.0
VENTANA_CRONICO_DIAS = 30
UMBRAL_CRONICO = 3

# Pre-calcular usuarios crónicos (≥3 rechazos en 30d basado en dataset completo)
from datetime import timedelta
df_rechazos["fecha_dt"] = pd.to_datetime(df_rechazos["fecha_hora"])
rechazos_por_usuario = df_rechazos.groupby("user_id").size().reset_index(name="n_rechazos")
USUARIOS_CRONICOS = set(rechazos_por_usuario[rechazos_por_usuario["n_rechazos"] >= UMBRAL_CRONICO]["user_id"])

def build_context_uc1(txn_id: str, user_id: str) -> dict:
    """
    Construye el payload JSON de contexto para Havi en UC1.
    
    Args:
        txn_id:  ID de la transacción rechazada (debe ser no_procesada con motivo UC1)
        user_id: ID del usuario afectado
    
    Returns:
        dict con el contexto estructurado listo para inyectar a Havi
    
    Raises:
        ValueError: si txn_id no es una transacción UC1 válida
    """
    txn_rows = df_tx[df_tx["transaccion_id"] == txn_id]
    if len(txn_rows) == 0:
        raise ValueError(f"Transacción {txn_id} no encontrada")
    txn = txn_rows.iloc[0]

    if str(txn.get("motivo_no_procesada", "")) not in MOTIVOS_UC1:
        raise ValueError(f"Transacción {txn_id} no es un rechazo UC1 (motivo: {txn.get('motivo_no_procesada')})")

    cli_rows = df_cli[df_cli["user_id"] == user_id]
    cli = cli_rows.iloc[0] if len(cli_rows) > 0 else pd.Series(dtype=object)

    prod_origen_rows = df_prod[df_prod["producto_id"] == txn["producto_id"]]
    prod_origen = prod_origen_rows.iloc[0] if len(prod_origen_rows) > 0 else None

    productos_usuario = df_prod[
        (df_prod["user_id"] == user_id) &
        (df_prod["estatus"] == "activo") &
        (df_prod["tipo_producto"].isin(PRIORIDAD_PRODUCTO.keys())) &
        (df_prod["saldo_actual"] >= float(txn["monto"])) &
        (df_prod["producto_id"] != txn["producto_id"])
    ].copy()

    prod_alt = None
    if len(productos_usuario) > 0:
        productos_usuario["_prioridad"] = productos_usuario["tipo_producto"].map(PRIORIDAD_PRODUCTO)
        prod_alt = productos_usuario.sort_values("_prioridad").iloc[0]

    saldo_origen = float(prod_origen["saldo_actual"]) if prod_origen is not None else 0.0
    monto_faltante = max(0.0, float(txn["monto"]) - saldo_origen)
    motivo = str(txn["motivo_no_procesada"])
    situacion = "rechazo_por_saldo" if motivo == "saldo_insuficiente" else "rechazo_por_limite"

    nombre = str(cli.get("nombre", "")) if "nombre" in cli.index else f"cliente {user_id}"

    return {
        "user_id":                      user_id,
        "transaccion_id":               txn_id,
        "situacion":                    situacion,
        "motivo":                       motivo,
        "monto_rechazado":              round(float(txn["monto"]), 2),
        "comercio":                     str(txn["comercio_nombre"]) if pd.notna(txn["comercio_nombre"]) else str(txn["categoria_mcc"]),
        "ciudad_transaccion":           str(txn["ciudad_transaccion"]),
        "fecha_hora":                   str(txn["fecha_hora"]),
        "es_internacional":             bool(txn["es_internacional"]),
        "nombre_usuario":               nombre,
        "saldo_actual_producto_origen": round(saldo_origen, 2),
        "monto_faltante":               round(monto_faltante, 2),
        "tiene_alternativo":            prod_alt is not None,
        "producto_alternativo":         str(prod_alt["tipo_producto"]) if prod_alt is not None else None,
        "producto_alternativo_id":      str(prod_alt["producto_id"]) if prod_alt is not None else None,
        "monto_disponible_alternativo": round(float(prod_alt["saldo_actual"]), 2) if prod_alt is not None else None,
        "es_cronico":                   user_id in USUARIOS_CRONICOS,
        "canal_preferido":              str(cli.get("preferencia_canal", "app_ios")) if "preferencia_canal" in cli.index else "app_ios",
    }

print(f"build_context_uc1 definida. Usuarios crónicos en dataset: {len(USUARIOS_CRONICOS)}")


build_context_uc1 definida. Usuarios crónicos en dataset: 265


## Mock: `transferFunds()`

In [5]:
def transferFunds(from_producto_id: str, to_producto_id: str, monto: float, user_id: str) -> dict:
    """
    Mock de la herramienta transferFunds().
    Transfiere fondos entre dos productos del mismo usuario para cubrir un rechazo.
    
    En producción: llamada al core bancario vía API interna.
    
    Args:
        from_producto_id: ID del producto con saldo disponible (el alternativo)
        to_producto_id:   ID del producto que fue rechazado (el origen)
        monto:            MXN a transferir
        user_id:          ID del usuario (seguridad)
    
    Returns:
        dict con {success, message, nueva_txn_id, estatus_resultante, error_code}
    
    Estados posibles de error:
        MONTO_BAJO_MINIMO    — monto < $50 MXN
        PRODUCTO_NO_ENCONTRADO
        PRODUCTO_NO_AUTORIZADO — producto no pertenece al user_id
        PRODUCTO_INACTIVO
        SALDO_INSUFICIENTE
        USUARIO_RECHAZA       — el usuario rechaza la sugerencia de transferencia
    """
    if monto < MONTO_MINIMO_MXN:
        return {"success": False, "message": f"Monto ${monto:.2f} menor al mínimo ${MONTO_MINIMO_MXN:.2f}",
                "nueva_txn_id": None, "estatus_resultante": None, "error_code": "MONTO_BAJO_MINIMO"}

    prod_from = df_prod[df_prod["producto_id"] == from_producto_id]
    if len(prod_from) == 0:
        return {"success": False, "message": "Producto origen no encontrado",
                "nueva_txn_id": None, "estatus_resultante": None, "error_code": "PRODUCTO_NO_ENCONTRADO"}
    prod_from = prod_from.iloc[0]

    if prod_from["user_id"] != user_id:
        return {"success": False, "message": "Producto no pertenece al usuario",
                "nueva_txn_id": None, "estatus_resultante": None, "error_code": "PRODUCTO_NO_AUTORIZADO"}

    if prod_from["estatus"] != "activo":
        return {"success": False, "message": f"Producto origen en estatus '{prod_from['estatus']}'",
                "nueva_txn_id": None, "estatus_resultante": None, "error_code": "PRODUCTO_INACTIVO"}

    if float(prod_from["saldo_actual"]) < monto:
        return {"success": False,
                "message": f"Saldo insuficiente: ${prod_from['saldo_actual']:.2f} < ${monto:.2f}",
                "nueva_txn_id": None, "estatus_resultante": None, "error_code": "SALDO_INSUFICIENTE"}

    nueva_txn_id = f"TXN-TRANSFER-{user_id[-5:]}-{int(monto)}-AUTO"
    return {
        "success":             True,
        "message":             f"Transferencia de ${monto:.2f} MXN desde {prod_from['tipo_producto']} completada.",
        "nueva_txn_id":        nueva_txn_id,
        "estatus_resultante":  "completada",
        "error_code":          None,
    }

print("transferFunds mock definido")


transferFunds mock definido


## Escenario 1: Rechazo por saldo insuficiente — con producto alternativo disponible

In [6]:
# Buscar un caso real: rechazo por saldo, con producto alternativo que cubre
candidatos_escenario1 = []
for _, row in df_rechazos[df_rechazos["motivo_no_procesada"] == "saldo_insuficiente"].head(300).iterrows():
    uid = row["user_id"]
    monto = float(row["monto"])
    otros_prods = df_prod[
        (df_prod["user_id"] == uid) &
        (df_prod["estatus"] == "activo") &
        (df_prod["tipo_producto"].isin(PRIORIDAD_PRODUCTO.keys())) &
        (df_prod["saldo_actual"] >= monto) &
        (df_prod["producto_id"] != row["producto_id"])
    ]
    if len(otros_prods) > 0 and monto >= MONTO_MINIMO_MXN:
        candidatos_escenario1.append(row)
        if len(candidatos_escenario1) >= 1:
            break

if candidatos_escenario1:
    txn_e1 = candidatos_escenario1[0]
    ctx_e1 = build_context_uc1(txn_e1["transaccion_id"], txn_e1["user_id"])
    print("=== CONTEXTO ESCENARIO 1 ===")
    print(json.dumps(ctx_e1, ensure_ascii=False, indent=2))
else:
    print("No se encontró candidato para escenario 1")
    ctx_e1 = None


=== CONTEXTO ESCENARIO 1 ===
{
  "user_id": "USR-00001",
  "transaccion_id": "TXN-0000000050",
  "situacion": "rechazo_por_saldo",
  "motivo": "saldo_insuficiente",
  "monto_rechazado": 349.0,
  "comercio": "NewsDigital MX",
  "ciudad_transaccion": "CDMX - Benito Juárez",
  "fecha_hora": "2025-08-23 20:37:31",
  "es_internacional": false,
  "nombre_usuario": "cliente USR-00001",
  "saldo_actual_producto_origen": 80954.6,
  "monto_faltante": 0.0,
  "tiene_alternativo": true,
  "producto_alternativo": "tarjeta_credito_hey",
  "producto_alternativo_id": "PRD-00000002",
  "monto_disponible_alternativo": 88790.4,
  "es_cronico": false,
  "canal_preferido": "app_android"
}


In [7]:
if ctx_e1:
    # Simular el flujo completo
    print("\n--- MENSAJE DE HAVI (texto) ---")
    nombre = ctx_e1.get("nombre_usuario", "")
    nombre_str = f"Hola {nombre}, " if nombre else "Hola, "
    print(f"""{nombre_str}tu compra de ${ctx_e1['monto_rechazado']:,.0f} en {ctx_e1['comercio']} fue rechazada
por saldo insuficiente. ¡Pero no te preocupes! Tienes ${ctx_e1['monto_disponible_alternativo']:,.0f} disponibles
en tu {ctx_e1['producto_alternativo'].replace('_',' ')}. ¿Quieres que transfiera
${ctx_e1['monto_rechazado']:,.0f} para completar tu compra?""")

    print("\n--- USUARIO ACEPTA → transferFunds() ---")
    result = transferFunds(
        from_producto_id = ctx_e1["producto_alternativo_id"],
        to_producto_id   = ctx_e1["transaccion_id"],  # mock destino
        monto            = ctx_e1["monto_rechazado"],
        user_id          = ctx_e1["user_id"]
    )
    print(json.dumps(result, ensure_ascii=False, indent=2))

    print("\n--- USUARIO RECHAZA LA SUGERENCIA ---")
    print("Havi: 'Entendido, no realizaré la transferencia. ¿Hay algo más en lo que pueda ayudarte?'")
    print("Acción backend: ninguna. Alerta marcada como 'rechazada_por_usuario'. No se vuelve a mostrar en 24h.")



--- MENSAJE DE HAVI (texto) ---
Hola cliente USR-00001, tu compra de $349 en NewsDigital MX fue rechazada
por saldo insuficiente. ¡Pero no te preocupes! Tienes $88,790 disponibles
en tu tarjeta credito hey. ¿Quieres que transfiera
$349 para completar tu compra?

--- USUARIO ACEPTA → transferFunds() ---
{
  "success": true,
  "message": "Transferencia de $349.00 MXN desde tarjeta_credito_hey completada.",
  "nueva_txn_id": "TXN-TRANSFER-00001-349-AUTO",
  "estatus_resultante": "completada",
  "error_code": null
}

--- USUARIO RECHAZA LA SUGERENCIA ---
Havi: 'Entendido, no realizaré la transferencia. ¿Hay algo más en lo que pueda ayudarte?'
Acción backend: ninguna. Alerta marcada como 'rechazada_por_usuario'. No se vuelve a mostrar en 24h.


## Escenario 2: Rechazo por límite excedido — TC casi al límite

In [8]:
candidatos_e2 = df_rechazos[df_rechazos["motivo_no_procesada"] == "limite_excedido"].head(100)
for _, row in candidatos_e2.iterrows():
    uid = row["user_id"]
    monto = float(row["monto"])
    otros = df_prod[
        (df_prod["user_id"] == uid) &
        (df_prod["estatus"] == "activo") &
        (df_prod["tipo_producto"].isin(PRIORIDAD_PRODUCTO.keys())) &
        (df_prod["saldo_actual"] >= monto) &
        (df_prod["producto_id"] != row["producto_id"])
    ]
    if len(otros) > 0 and monto >= MONTO_MINIMO_MXN:
        txn_e2 = row
        ctx_e2 = build_context_uc1(txn_e2["transaccion_id"], txn_e2["user_id"])
        break
else:
    # fallback: usar cualquier rechazo de límite
    txn_e2 = candidatos_e2.iloc[0]
    ctx_e2 = build_context_uc1(txn_e2["transaccion_id"], txn_e2["user_id"])

print("=== CONTEXTO ESCENARIO 2 ===")
print(json.dumps(ctx_e2, ensure_ascii=False, indent=2))


=== CONTEXTO ESCENARIO 2 ===
{
  "user_id": "USR-00006",
  "transaccion_id": "TXN-0000000417",
  "situacion": "rechazo_por_limite",
  "motivo": "limite_excedido",
  "monto_rechazado": 499.0,
  "comercio": "GamerPass",
  "ciudad_transaccion": "Zacatecas",
  "fecha_hora": "2025-08-15 12:57:17",
  "es_internacional": false,
  "nombre_usuario": "cliente USR-00006",
  "saldo_actual_producto_origen": 72254.55,
  "monto_faltante": 0.0,
  "tiene_alternativo": true,
  "producto_alternativo": "tarjeta_credito_hey",
  "producto_alternativo_id": "PRD-00000013",
  "monto_disponible_alternativo": 15132.9,
  "es_cronico": false,
  "canal_preferido": "app_android"
}


In [9]:
print("--- MENSAJE DE HAVI (texto) ---")
nombre2 = ctx_e2.get("nombre_usuario", "")
nombre_str2 = f"Hola {nombre2}, " if nombre2 else "Hola, "
if ctx_e2["tiene_alternativo"]:
    print(f"""{nombre_str2}tu compra de ${ctx_e2['monto_rechazado']:,.0f} en {ctx_e2['comercio']} fue rechazada
porque tu tarjeta alcanzó su límite. Tienes ${ctx_e2['monto_disponible_alternativo']:,.0f} disponibles
en tu {ctx_e2['producto_alternativo'].replace('_',' ')}. ¿Te lo cargo ahí?""")
else:
    print(f"""{nombre_str2}tu compra de ${ctx_e2['monto_rechazado']:,.0f} en {ctx_e2['comercio']} fue rechazada
por límite de tarjeta. Para liberar línea, puedes hacer un pago a tu tarjeta de crédito.
¿Quieres que te muestre las opciones de pago?""")


--- MENSAJE DE HAVI (texto) ---
Hola cliente USR-00006, tu compra de $499 en GamerPass fue rechazada
porque tu tarjeta alcanzó su límite. Tienes $15,133 disponibles
en tu tarjeta credito hey. ¿Te lo cargo ahí?


## Escenario 3: Usuario crónico — ≥3 rechazos, saldo crítico

In [10]:
# Buscar usuario crónico con rechazo reciente
usuarios_cronicos_con_rechazos = df_rechazos[df_rechazos["user_id"].isin(USUARIOS_CRONICOS)]
if len(usuarios_cronicos_con_rechazos) > 0:
    txn_e3 = usuarios_cronicos_con_rechazos.iloc[0]
    ctx_e3 = build_context_uc1(txn_e3["transaccion_id"], txn_e3["user_id"])
    print(f"Usuario crónico encontrado: {ctx_e3['user_id']} (es_cronico={ctx_e3['es_cronico']})")
    print(json.dumps(ctx_e3, ensure_ascii=False, indent=2))
else:
    # fallback: tomar cualquier rechazo
    txn_e3 = df_rechazos.iloc[10]
    ctx_e3 = build_context_uc1(txn_e3["transaccion_id"], txn_e3["user_id"])
    ctx_e3["es_cronico"] = True  # forzar para demo
    print("=== CONTEXTO ESCENARIO 3 (usuario crónico simulado) ===")
    print(json.dumps(ctx_e3, ensure_ascii=False, indent=2))


Usuario crónico encontrado: USR-00031 (es_cronico=True)
{
  "user_id": "USR-00031",
  "transaccion_id": "TXN-0000001798",
  "situacion": "rechazo_por_limite",
  "motivo": "limite_excedido",
  "monto_rechazado": 5300.0,
  "comercio": "transferencia",
  "ciudad_transaccion": "CDMX - Miguel Hidalgo",
  "fecha_hora": "2025-03-12 16:49:45",
  "es_internacional": false,
  "nombre_usuario": "cliente USR-00031",
  "saldo_actual_producto_origen": 36582.6,
  "monto_faltante": 0.0,
  "tiene_alternativo": true,
  "producto_alternativo": "cuenta_debito",
  "producto_alternativo_id": "PRD-00000078",
  "monto_disponible_alternativo": 13909.78,
  "es_cronico": true,
  "canal_preferido": "app_ios"
}


In [11]:
print("--- MENSAJE DE HAVI (crónico — tono más empático y propositivo) ---")
nombre3 = ctx_e3.get("nombre_usuario", "")
nombre_str3 = f"{nombre3}, " if nombre3 else ""
print(f"""Oye {nombre_str3}noto que has tenido varios rechazos últimamente.
Tu compra de ${ctx_e3['monto_rechazado']:,.0f} en {ctx_e3['comercio']} fue rechazada nuevamente.
Puede que valga la pena revisar cómo está distribuido tu dinero entre tus productos.
¿Te explico cómo configurar tu cuenta para que esto no vuelva a pasar?""")


--- MENSAJE DE HAVI (crónico — tono más empático y propositivo) ---
Oye cliente USR-00031, noto que has tenido varios rechazos últimamente.
Tu compra de $5,300 en transferencia fue rechazada nuevamente.
Puede que valga la pena revisar cómo está distribuido tu dinero entre tus productos.
¿Te explico cómo configurar tu cuenta para que esto no vuelva a pasar?


## Documentación: ¿Qué pasa si el usuario rechaza la transferencia?

In [12]:
RECHAZO_USUARIO_POLICY = {
    "trigger": "El usuario responde 'no', 'no gracias', 'déjalo', 'no hagas nada' o similar",
    "accion_inmediata": "No se ejecuta transferFunds(). Alerta se marca como 'rechazada_por_usuario'.",
    "cooldown_horas": 24,
    "regla": "No mostrar la misma sugerencia de transferencia para el mismo usuario en las próximas 24h.",
    "excepciones": [
        "Si el usuario tiene es_cronico=True, mostrar un mensaje diferente (asesoría presupuestal) en lugar de solo silenciar.",
        "Si el mismo comercio genera otro rechazo dentro del cooldown, sí notificar con un mensaje diferente."
    ],
    "estado_alerta_resultante": "rechazada_por_usuario",
    "feedback_al_modelo": "Incrementar penalización de scoring para alertas rechazadas por este usuario."
}

print("Política de rechazo documentada:")
print(json.dumps(RECHAZO_USUARIO_POLICY, ensure_ascii=False, indent=2))


Política de rechazo documentada:
{
  "trigger": "El usuario responde 'no', 'no gracias', 'déjalo', 'no hagas nada' o similar",
  "accion_inmediata": "No se ejecuta transferFunds(). Alerta se marca como 'rechazada_por_usuario'.",
  "cooldown_horas": 24,
  "regla": "No mostrar la misma sugerencia de transferencia para el mismo usuario en las próximas 24h.",
  "excepciones": [
    "Si el usuario tiene es_cronico=True, mostrar un mensaje diferente (asesoría presupuestal) en lugar de solo silenciar.",
    "Si el mismo comercio genera otro rechazo dentro del cooldown, sí notificar con un mensaje diferente."
  ],
  "estado_alerta_resultante": "rechazada_por_usuario",
  "feedback_al_modelo": "Incrementar penalización de scoring para alertas rechazadas por este usuario."
}


## Guardar outputs

In [13]:
output = {
    "fecha_generacion": datetime.utcnow().isoformat() + "Z",
    "uc": "UC1",
    "descripcion": "Contexto JSON para Havi — Financial Copilot (rechazo proactivo)",
    "schema": UC1_CONTEXT_SCHEMA,
    "escenarios": [
        {
            "id": "escenario_1_saldo",
            "descripcion": "Rechazo por saldo insuficiente con alternativo disponible",
            "contexto": ctx_e1,
            "tool_call": {
                "funcion": "transferFunds",
                "parametros": {
                    "from_producto_id": ctx_e1["producto_alternativo_id"] if ctx_e1 else None,
                    "to_producto_id": ctx_e1.get("transaccion_id") if ctx_e1 else None,
                    "monto": ctx_e1["monto_rechazado"] if ctx_e1 else None,
                    "user_id": ctx_e1["user_id"] if ctx_e1 else None
                }
            }
        },
        {
            "id": "escenario_2_limite",
            "descripcion": "Rechazo por límite excedido",
            "contexto": ctx_e2,
        },
        {
            "id": "escenario_3_cronico",
            "descripcion": "Usuario crónico — tono empático y propositivo",
            "contexto": ctx_e3,
        },
    ],
    "politica_rechazo_usuario": RECHAZO_USUARIO_POLICY,
    "criterios_aceptacion": {
        "payload_json_con_schema": True,
        "tool_transferFunds_mockeada": True,
        "n_ejemplos_conversacion": 3,
        "rechazo_usuario_documentado": True,
    }
}

output_path = BASE_OUT / "uc1_integration_output.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2, default=str)

print(f"Output guardado en {output_path}")
print("\n✅ UC1 Integration — todos los criterios de aceptación cumplidos")


Output guardado en outputs\integration\uc1_integration_output.json

✅ UC1 Integration — todos los criterios de aceptación cumplidos
